In [67]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# Load the data
data = pd.read_csv('dataste.csv')

# Display the first few rows to understand the structure
data.head()

,Cost Rank,Product Name,Product Life Cycle,FY22 Q2,FY22 Q3,FY22 Q4,FY23 Q1,FY23 Q2,FY23 Q3,FY23 Q4,FY24 Q1,FY24 Q2,FY24 Q3,FY24 Q4,FY25 Q1,Your Forecast FY25 Q2
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,27302,22084,21830,29404,24518,NaN
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,5956,8450,8584,12552,9665,NaN
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,7295,8084,9249,10742,9189,NaN
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,16095,26125,24337,21988,32768,NaN
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,69599,78130,76840,85421,54494,NaN


In [11]:
import pandas as pd

# Load dataset
file_path = "dataste.csv"
df = pd.read_csv(file_path)

# Display all column names
print(df.columns)


Index(['Cost Rank', 'Product Name', 'Product Life Cycle', 'FY22 Q2', 'FY22 Q3',
       'FY22 Q4', 'FY23 Q1', 'FY23 Q2', 'FY23 Q3', 'FY23 Q4', 'FY24 Q1',
       'FY24 Q2', 'FY24 Q3', 'FY24 Q4', 'FY25 Q1', 'Your Forecast FY25 Q2'],
      dtype='object')


In [55]:
df.head()

,Cost Rank,Product Name,Product Life Cycle,FY22 Q2,FY22 Q3,FY22 Q4,FY23 Q1,FY23 Q2,FY23 Q3,FY23 Q4,FY24 Q1,FY24 Q2,FY24 Q3,FY24 Q4,FY25 Q1,Your Forecast FY25 Q2
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,27302,22084,21830,29404,24518,NaN
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,5956,8450,8584,12552,9665,NaN
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,7295,8084,9249,10742,9189,NaN
3,4,SWITCH Enterprise Low,Sustaining,24362.0,21308.0,1227.0,24186.0,7680.0,16772,17554,16095,26125,24337,21988,32768,NaN
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,69599,78130,76840,85421,54494,NaN


In [69]:
# Convert the quarters into a proper date format
def convert_quarter_to_date(quarter_str):
    if pd.isna(quarter_str):
        return np.nan
    
    fiscal_year = int('20' + quarter_str[2:4])
    quarter_num = int(quarter_str[-1])
    
    # Map fiscal quarters to calendar quarters (assuming fiscal year starts 3 months after calendar year)
    calendar_year = fiscal_year
    calendar_quarter = quarter_num
    
    # Use the middle month of each quarter
    month = 3 * calendar_quarter - 2
    return pd.Timestamp(f"{calendar_year}-{month:02d}-15")

# Clean column names and extract time series data
df = data.copy()
df.columns = [col.strip() for col in df.columns]
time_cols = [col for col in df.columns if col.startswith('FY')]

# Reshape data for time series analysis
products_df = pd.DataFrame()
for idx, row in df.iterrows():
    product_data = {
        'Cost_Rank': row['Cost Rank'],
        'Product_Name': row['Product Name'],
        'Life_Cycle': row['Product Life Cycle']
    }
    
    for col in time_cols:
        if col != 'Your Forecast FY25 Q2':
            products_df = pd.concat([products_df, pd.DataFrame({
                **product_data,
                'Quarter': col,
                'Units': row[col]
            }, index=[0])], ignore_index=True)

# Check the transformed data
products_df.head()

,Cost_Rank,Product_Name,Life_Cycle,Quarter,Units
0,1,SWITCH Enterprise High,Sustaining,FY22 Q2,57147.0
1,1,SWITCH Enterprise High,Sustaining,FY22 Q3,52873.0
2,1,SWITCH Enterprise High,Sustaining,FY22 Q4,52870.0
3,1,SWITCH Enterprise High,Sustaining,FY23 Q1,38833.0
4,1,SWITCH Enterprise High,Sustaining,FY23 Q2,27114.0


In [71]:
# Group by product and prepare for forecasting
product_groups = products_df.groupby(['Cost_Rank', 'Product_Name'])

# Set of forecasting methods to try
forecasts = {}

for name, group in product_groups:
    product_id = name[0]  # Cost_Rank
    product_name = name[1]
    
    # Sort by quarter for time series analysis
    group = group.sort_values('Quarter')
    
    # Extract the time series
    time_series = group['Units'].values
    
    # Skip products with too few data points
    if len(time_series) < 4:
        continue
    
    # Check for NaN values
    valid_values = time_series[~np.isnan(time_series)]
    
    if len(valid_values) < 4:
        # Not enough data points, use average of available data
        forecasts[product_id] = np.mean(valid_values)
        continue
    
    # Method 1: Simple Moving Average (last 4 quarters)
    ma_forecast = np.mean(valid_values[-4:])
    
    # Method 2: Weighted Moving Average (higher weight to recent quarters)
    weights = np.array([0.1, 0.2, 0.3, 0.4])
    wma_forecast = np.sum(weights * valid_values[-4:]) / np.sum(weights)
    
    # Method 3: Year-over-Year growth rate (comparing with same quarter last year)
    if len(valid_values) >= 5:
        yoy_rate = valid_values[-1] / valid_values[-5] if valid_values[-5] > 0 else 1
        yoy_forecast = valid_values[-1] * yoy_rate
    else:
        yoy_forecast = valid_values[-1]
    
    # Method 4: Exponential smoothing
    try:
        # Fill NaN with interpolation
        filled_series = pd.Series(time_series).interpolate().values
        model = ExponentialSmoothing(
            filled_series, 
            trend='add',
            seasonal='add', 
            seasonal_periods=4
        ).fit()
        hw_forecast = model.forecast(1)[0]
    except:
        hw_forecast = ma_forecast
    
    # Combine forecasts with weights
    # Weight more on YoY and Holt-Winters for seasonal products
    # Weight more on recent performance (MA, WMA) for sustaining products
    if group['Life_Cycle'].iloc[0] == 'Sustaining':
        combined_forecast = (0.3 * ma_forecast + 
                             0.3 * wma_forecast + 
                             0.2 * yoy_forecast + 
                             0.2 * hw_forecast)
    elif group['Life_Cycle'].iloc[0] == 'Decline':
        combined_forecast = (0.2 * ma_forecast + 
                             0.2 * wma_forecast + 
                             0.4 * yoy_forecast + 
                             0.2 * hw_forecast)
    else:  # NPI - New Product Introduction
        combined_forecast = (0.1 * ma_forecast + 
                             0.3 * wma_forecast + 
                             0.1 * yoy_forecast + 
                             0.5 * hw_forecast)
    
    # Ensure forecast is not negative
    forecasts[product_id] = max(0, round(combined_forecast))

ValueError: cannot convert float NaN to integer